# pyGLAM against the R implementations

Cross-validation of `pyglam` against the established R packages `gld` and `GLDEX`, in two parts:

- **A. Functions.** For a set of parameter vectors of both parameterizations, the quantile
  function, the CDF, the density and the first four moments of `pyglam` are compared with
  `gld::qgl`, `gld::pgl`, `gld::dgl` and `GLDEX::fun.theo.mv.gld`. Same parameters, same function:
  the numbers must agree to numerical precision.
- **B. Fits.** The same samples are fitted by moment matching with `pyglam` and with
  `GLDEX::fun.RMFMKL.mm` (FKML) and `GLDEX::fun.RPRS.mm` (RS). Moment matching has several roots,
  so the parameters are not expected to be identical; the comparison is on the fitted
  distributions: validity, distance to the analytical target, reproduced moments, agreement between
  the two fitted quantile functions, and time.

Requires R with the packages `gld` and `GLDEX` (`install.packages(c("gld", "GLDEX"))`). The
`Rscript` executable is found on the `PATH`, in the `RSCRIPT` environment variable, or in the
default Windows installation folder.

## **1. Libraries and R bridge**

In [1]:
%matplotlib inline
import glob
import os
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

REPO = Path.cwd()
sys.path.insert(0, str(REPO))
os.environ['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO), os.environ.get('PYTHONPATH')]))

import numpy as np
import pandas as pd
import scipy.stats as st
from joblib import Parallel, delayed
from scipy.integrate import trapezoid

import pyglam as glam


def find_rscript():
    candidates = [os.environ.get('RSCRIPT'), shutil.which('Rscript')]
    candidates += sorted(glob.glob(r'C:\Program Files\R\R-*\bin\Rscript.exe'))[::-1]
    for path in candidates:
        if path and Path(path).exists():
            return path
    raise FileNotFoundError('Rscript not found: install R or set the RSCRIPT environment variable.')


RSCRIPT = find_rscript()
WORK    = Path(tempfile.mkdtemp(prefix='pyglam_vs_gld_'))


def run_r(script):
    # Run an R script in the working folder, where the CSV files are exchanged
    path = WORK / 'script.R'
    path.write_text(script, encoding='utf-8')
    done = subprocess.run([RSCRIPT, '--vanilla', str(path)], cwd=WORK, capture_output=True,
                          text=True, encoding='utf-8', errors='replace')
    if done.returncode != 0:
        raise RuntimeError(done.stderr[-3000:])
    return done.stdout


print(run_r('cat(R.version.string, "| gld", as.character(packageVersion("gld")), '
            '"| GLDEX", as.character(packageVersion("GLDEX")), "\\n")'))

R version 4.6.1 (2026-06-24 ucrt) | gld 2.6.8 | GLDEX 2.0.0.9.4 



## **2. Quantile functions written from the definitions**

Validity and distances of the fitted parameters are computed from the quantile functions written
directly from their definitions, so that the parameters returned by either implementation are
judged by the same code, including parameters that `pyglam` itself would refuse.

In [2]:
U = np.linspace(1e-6, 1.0 - 1e-6, 4001)
TIE = 1e-10


def q_fkml(u, l1, l2, l3, l4):
    def phi(v, lam):
        return np.log(v) if lam == 0 else np.expm1(lam * np.log(v)) / lam
    return l1 + (phi(u, l3) - phi(1.0 - u, l4)) / l2


def q_rs(u, l1, l2, l3, l4):
    return l1 + (u ** l3 - (1.0 - u) ** l4) / l2


QUANTILE = {'FKML': q_fkml, 'RS': q_rs}


def quantiles(family, params, u=U):
    with np.errstate(all='ignore'):
        return QUANTILE[family](u, *params)


def is_valid(x):
    # a distribution: finite and non-decreasing on the probability grid
    return bool(np.all(np.isfinite(x)) and np.all(np.diff(x) >= -TIE * max(1.0, np.abs(x).max())))

## **3. Part A: the same parameters, the same functions**

Parameter vectors near the fits of the numerical study, plus asymmetric and heavy-tailed cases, in
every sign configuration the RS form admits within the moment domain.

In [3]:
param_sets = {
    'FKML': [(0.004, 1.465, 0.133, 0.138), (0.5, 2.0, 1.0, 1.0), (0.288, 1.162, 0.444, -0.027),
             (-0.005, 1.712, -0.136, -0.145), (0.0, 1.0, 0.5, 2.0), (1.0, 0.5, -0.2, 0.3)],
    'RS':   [(0.017, 0.198, 0.137, 0.133), (0.5, 2.0, 1.0, 1.0), (0.0, 1.0, 0.1, 0.5),
             (0.0, -0.25, -0.14, -0.14), (0.0, -0.2, -0.05, -0.15), (0.0, -1.0, -0.249, 8.0)],
}
classes = {'FKML': glam.GlamFKML, 'RS': glam.GlamRS}

u_a  = np.linspace(0.001, 0.999, 199)
rows = []
for family, sets in param_sets.items():
    for k, params in enumerate(sets):
        model = classes[family](*params)
        if family == 'RS':
            assert model._rs_region() is not None, params
        x = np.asarray(model.ppf(u_a), dtype=float)
        rows.append({'family': family, 'set': k, 'params': params, 'x': x,
                     'F': np.asarray(model.cdf(x), dtype=float),
                     'f': np.asarray(model.pdf(x), dtype=float),
                     'moments': np.asarray(model._theoretical_moments(*params), dtype=float)})

# hand the parameters and the evaluation points to R
pd.DataFrame([(r['family'], r['set'], *r['params']) for r in rows],
             columns=['family', 'set', 'l1', 'l2', 'l3', 'l4']).to_csv(WORK / 'params.csv', index=False)
pd.DataFrame({'u': u_a}).to_csv(WORK / 'u.csv', index=False)
pd.DataFrame(np.column_stack([r['x'] for r in rows])).to_csv(WORK / 'x.csv', index=False)

run_r(r'''
suppressMessages({library(gld); library(GLDEX)})
p <- read.csv("params.csv"); u <- read.csv("u.csv")$u; x <- read.csv("x.csv")
q <- f <- d <- matrix(NA, length(u), nrow(p)); m <- matrix(NA, nrow(p), 4)
for (i in seq_len(nrow(p))) {
  l  <- as.numeric(p[i, c("l1", "l2", "l3", "l4")])
  pr <- if (p$family[i] == "FKML") "fkml" else "rs"
  q[, i] <- qgl(u, l, param = pr)
  f[, i] <- pgl(x[[i]], l, param = pr)
  d[, i] <- dgl(x[[i]], l, param = pr)
  m[i, ] <- fun.theo.mv.gld(l[1], l[2], l[3], l[4], param = if (pr == "fkml") "fmkl" else "rs")
}
write.csv(q, "ppf_r.csv", row.names = FALSE); write.csv(f, "cdf_r.csv", row.names = FALSE)
write.csv(d, "pdf_r.csv", row.names = FALSE); write.csv(m, "moments_r.csv", row.names = FALSE)
''')

q_r = pd.read_csv(WORK / 'ppf_r.csv').to_numpy()
F_r = pd.read_csv(WORK / 'cdf_r.csv').to_numpy()
f_r = pd.read_csv(WORK / 'pdf_r.csv').to_numpy()
m_r = pd.read_csv(WORK / 'moments_r.csv').to_numpy()

functions = []
for i, r in enumerate(rows):
    span = r['x'][-1] - r['x'][0]
    m_py, m_R = r['moments'], m_r[i]
    functions.append({
        'Family': r['family'], 'lambda': r['params'],
        'quantile': np.max(np.abs(r['x'] - q_r[:, i])) / span,
        'cdf': np.max(np.abs(r['F'] - F_r[:, i])),
        'pdf': np.max(np.abs(r['f'] - f_r[:, i])) / np.max(np.abs(f_r[:, i])),
        'mean': abs(m_py[0] - m_R[0]) / np.sqrt(m_R[1]),
        'variance': abs(m_py[1] - m_R[1]) / m_R[1],
        'skewness': abs(m_py[2] - m_R[2]),
        'kurtosis': abs(m_py[3] - m_R[3]),
    })
functions = pd.DataFrame(functions)
with pd.option_context('display.float_format', '{:.1e}'.format):
    display(functions)

,Family,lambda,quantile,cdf,pdf,mean,variance,skewness,kurtosis
0,FKML,"(0.004, 1.465, 0.133, 0.138)",9.1e-16,5.2e-09,1.2e-08,5.9e-18,2.2e-16,4.9e-14,7.3e-13
1,FKML,"(0.5, 2.0, 1.0, 1.0)",5.0e-16,1.0e-08,0.0e+00,0.0e+00,3.3e-16,0.0e+00,2.2e-16
2,FKML,"(0.288, 1.162, 0.444, -0.027)",9.5e-16,2.5e-13,8.4e-13,1.7e-16,6.8e-16,2.7e-15,8.9e-16
3,FKML,"(-0.005, 1.712, -0.136, -0.145)",3.9e-16,2.1e-13,6.5e-13,5.1e-17,3.3e-15,3.2e-14,1.7e-13
4,FKML,"(0.0, 1.0, 0.5, 2.0)",2.0e-15,2.5e-13,2.8e-12,5.4e-16,1.4e-15,4.1e-15,1.4e-14
5,FKML,"(1.0, 0.5, -0.2, 0.3)",1.1e-15,9.3e-09,5.8e-09,6.8e-18,5.7e-15,7.1e-14,1.5e-12
6,RS,"(0.017, 0.198, 0.137, 0.133)",8.0e-16,6.9e-09,9.4e-09,1.1e-17,5.4e-15,2.9e-14,3.4e-13
7,RS,"(0.5, 2.0, 1.0, 1.0)",5.0e-16,1.0e-08,0.0e+00,0.0e+00,3.3e-16,0.0e+00,2.2e-16
8,RS,"(0.0, 1.0, 0.1, 0.5)",3.4e-16,8.0e-09,4.2e-09,1.4e-15,1.9e-15,6.2e-15,5.0e-14
9,RS,"(0.0, -0.25, -0.14, -0.14)",3.7e-16,7.3e-09,3.6e-09,0.0e+00,5.3e-15,0.0e+00,4.3e-13


The columns are the largest discrepancy over the evaluation grid: the quantile relative to the
range of the evaluated quantiles, the CDF in probability, the density relative to its maximum, the
mean in standard deviations, the variance relatively, and skewness and kurtosis in absolute terms.

## **4. Part B: fitting the same samples**

In [4]:
targets = {
    'Normal':    st.norm(loc=0.0, scale=1.0),
    'Gumbel':    st.gumbel_r(loc=0.0, scale=1.0),
    'Uniform':   st.uniform(loc=0.0, scale=1.0),
    'Student-t': st.t(df=4, loc=0.0, scale=1.0),
    'Gamma':     st.gamma(a=2.0, scale=1.0),
    'Beta':      st.beta(a=2.0, b=5.0),
}
sizes, n_rep, n_starts, base_seed = (1000, 5000), 50, 15, 7
n_jobs = max(1, (os.cpu_count() or 2) - 1)

samples = []
for t, (name, dist) in enumerate(targets.items()):
    for n in sizes:
        for rep in range(n_rep):
            seed = int(np.random.SeedSequence([base_seed, t, n, rep]).generate_state(1)[0])
            samples.append({'id': len(samples), 'Distribution': name, 'N': n, 'rep': rep,
                            'x': dist.rvs(size=n, random_state=np.random.default_rng(seed))})

print(len(samples), 'samples')

600 samples


In [5]:
def fit_pyglam(s, family):
    cls = classes[family]
    t0  = time.perf_counter()
    sol = cls().fit_lambdas(s['x'], method='least_squares', n_starts=n_starts, seed=s['id'])
    return {'id': s['id'], 'Family': family, 'tool': 'pyGLAM', 'time': time.perf_counter() - t0,
            'l1': sol.x[0], 'l2': sol.x[1], 'l3': sol.x[2], 'l4': sol.x[3]}


t0 = time.perf_counter()
py_fits = Parallel(n_jobs=n_jobs)(delayed(fit_pyglam)(s, f) for s in samples for f in classes)
print(f'pyGLAM: {time.perf_counter() - t0:.1f} s of wall time')

# GLDEX fits one sample in seconds, so the samples are split into chunks fitted by parallel R
# processes; each process writes its own file
R_FIT = r'''
suppressMessages(library(GLDEX))
d   <- read.csv("@IN@")
out <- list()
for (i in unique(d$id)) {
  x <- d$x[d$id == i]
  for (fam in c("FKML", "RS")) {
    set.seed(i)
    t0 <- proc.time()[["elapsed"]]
    l  <- tryCatch(if (fam == "FKML") fun.RMFMKL.mm(x) else fun.RPRS.mm(x),
                   error = function(e) rep(NA, 4))
    el <- proc.time()[["elapsed"]] - t0
    out[[length(out) + 1]] <- data.frame(id = i, Family = fam, time = el,
                                         l1 = l[1], l2 = l[2], l3 = l[3], l4 = l[4])
  }
}
write.csv(do.call(rbind, out), "@OUT@", row.names = FALSE)
'''

chunks = np.array_split(np.arange(len(samples)), n_jobs)
t0 = time.perf_counter()
for k, ids in enumerate(chunks):
    pd.DataFrame({'id': np.concatenate([[i] * samples[i]['N'] for i in ids]),
                  'x': np.concatenate([samples[i]['x'] for i in ids])}).to_csv(WORK / f'samples_{k}.csv', index=False)
    (WORK / f'fit_{k}.R').write_text(R_FIT.replace('@IN@', f'samples_{k}.csv')
                                          .replace('@OUT@', f'gldex_{k}.csv'), encoding='utf-8')

# an R process occasionally dies without a message when many run at once; the fits are seeded,
# so a chunk that failed is simply run again
pending, errors = list(range(len(chunks))), {}
for attempt in range(3):
    procs = {k: subprocess.Popen([RSCRIPT, '--vanilla', f'fit_{k}.R'], cwd=WORK,
                                 stdout=subprocess.DEVNULL, stderr=subprocess.PIPE) for k in pending}
    for k, p in procs.items():
        _, err = p.communicate()
        errors[k] = err.decode('utf-8', 'replace')[-3000:]
    pending = [k for k, p in procs.items() if p.returncode != 0 or not (WORK / f'gldex_{k}.csv').exists()]
    if not pending:
        break
if pending:
    raise RuntimeError(f'R chunks {pending} failed: ' + errors[pending[0]])
print(f'GLDEX: {time.perf_counter() - t0:.1f} s of wall time on {len(chunks)} R processes')

r_fits = pd.concat([pd.read_csv(WORK / f'gldex_{k}.csv') for k in range(len(chunks))]).assign(tool='GLDEX')
fits   = pd.concat([pd.DataFrame(py_fits), r_fits], ignore_index=True)

pyGLAM: 12.5 s of wall time


GLDEX: 202.7 s of wall time on 31 R processes


Every fit is scored with the same code: validity of the quantile function, KS distance and
Wasserstein distance to the analytical target (the latter from the quantile functions, normalised
by the spread between $P_5$ and $P_{95}$ of the target), and the largest discrepancy between the
four moments it implies and the sample moments (variance with divisor $N$), with the mean and the
variance in relative terms. The implied moments are computed from the closed forms of `pyglam`,
which Part A validated against `GLDEX`.

In [6]:
by_id = {s['id']: s for s in samples}


def score(row):
    s, family = by_id[row['id']], row['Family']
    params    = (row['l1'], row['l2'], row['l3'], row['l4'])
    dist      = targets[s['Distribution']]
    out = {'Distribution': s['Distribution'], 'N': s['N']}
    if not np.all(np.isfinite(params)):
        return {**out, 'valid': False}
    x = quantiles(family, params)
    if not is_valid(x):
        return {**out, 'valid': False}
    spread = dist.ppf(0.95) - dist.ppf(0.05)
    cdf    = dist.cdf(x)
    ks     = max(np.max(np.abs(cdf - U)), cdf[0], 1.0 - cdf[-1])
    inner  = (U > 1e-4) & (U < 1 - 1e-4)
    w1     = trapezoid(np.abs(x[inner] - dist.ppf(U[inner])), U[inner]) / spread

    with np.errstate(all='ignore'):
        m = np.asarray(classes[family]()._theoretical_moments(*params), dtype=float)
    xs = s['x']; mu = xs.mean(); var = xs.var()
    sk = np.mean((xs - mu) ** 3) / var ** 1.5; ku = np.mean((xs - mu) ** 4) / var ** 2
    moment_gap = max(abs(m[0] - mu) / np.sqrt(var), abs(m[1] - var) / var, abs(m[2] - sk), abs(m[3] - ku))
    return {**out, 'valid': True, 'KS': ks, 'W1': w1, 'moment gap': moment_gap}


scored = pd.concat([fits, pd.DataFrame([score(r) for _, r in fits.iterrows()])], axis=1)


def agreement(group):
    # largest gap between the two fitted quantile functions on the same sample, on [0.01, 0.99]
    sample_id, family = group.name
    rows = {r['tool']: r for _, r in group.iterrows()}
    if not (rows['pyGLAM']['valid'] and rows['GLDEX']['valid']):
        return np.nan
    s = by_id[sample_id]
    dist  = targets[s['Distribution']]
    u     = np.linspace(0.01, 0.99, 197)
    q_py  = quantiles(family, rows['pyGLAM'][['l1', 'l2', 'l3', 'l4']].to_numpy(float), u)
    q_R   = quantiles(family, rows['GLDEX'][['l1', 'l2', 'l3', 'l4']].to_numpy(float), u)
    return np.max(np.abs(q_py - q_R)) / (dist.ppf(0.95) - dist.ppf(0.05))


agree = (scored.groupby(['id', 'Family']).apply(agreement, include_groups=False)
               .rename('quantile gap').reset_index())
agree = agree.merge(scored[['id', 'Distribution', 'N']].drop_duplicates(), on='id')

In [7]:
keys = ['Family', 'Distribution', 'N']
valid_rate = scored.groupby(keys + ['tool'], sort=False)['valid'].mean().unstack('tool') * 100
medians    = (scored[scored['valid']].groupby(keys + ['tool'], sort=False)[['KS', 'W1', 'moment gap']]
                    .median().unstack('tool'))
gap        = agree.groupby(keys, sort=False)['quantile gap'].median()
timing     = scored.groupby(['Family', 'tool'])['time'].agg(['mean', 'median', 'std']).round(3)

comparison = pd.concat({'valid (%)': valid_rate, 'KS': medians['KS'], 'W1': medians['W1'],
                        'moment gap': medians['moment gap']}, axis=1)
comparison[('quantile gap', 'median')] = gap
display(comparison.round(4))
display(timing)

valid (%)             KS              W1          \
tool                        pyGLAM  GLDEX  pyGLAM   GLDEX  pyGLAM   GLDEX   
Family Distribution N                                                       
FKML   Normal       1000     100.0  100.0  0.0124  0.0124  0.0090  0.0090   
                    5000     100.0  100.0  0.0057  0.0057  0.0041  0.0041   
       Gumbel       1000     100.0  100.0  0.0261  0.0230  0.0154  0.0143   
                    5000     100.0  100.0  0.0146  0.0187  0.0078  0.0099   
       Uniform      1000     100.0  100.0  0.0142  0.0150  0.0086  0.0100   
                    5000     100.0  100.0  0.0067  0.0076  0.0037  0.0043   
       Student-t    1000     100.0  100.0  0.0180  0.0180  0.0148  0.0148   
                    5000     100.0  100.0  0.0101  0.0101  0.0090  0.0090   
       Gamma        1000     100.0  100.0  0.0413  0.0250  0.0141  0.0131   
                    5000     100.0  100.0  0.0313  0.0243  0.0083  0.0098   
       Beta         1000     100.0  100.0  0.0171  0.0290  0.0100  0.0187   
                    5000     100.0  100.0  0.0096  0.0283  0.0048  0.0173   
RS     Normal       1000     100.0  100.0  0.0138  0.0138  0.0094  0.0094   
                    5000     100.0  100.0  0.0060  0.0060  0.0040  0.0040   
       Gumbel       1000     100.0  100.0  0.0200  0.0302  0.0135  0.0197   
                    5000     100.0  100.0  0.0110  0.0121  0.0063  0.0068   
       Uniform      1000     100.0  100.0  0.0144  0.0149  0.0086  0.0097   
                    5000     100.0  100.0  0.0067  0.0076  0.0037  0.0047   
       Student-t    1000     100.0  100.0  0.0195  0.0272  0.0160  0.0190   
                    5000     100.0  100.0  0.0122  0.0122  0.0101  0.0101   
       Gamma        1000     100.0  100.0  0.0292  0.0252  0.0192  0.0147   
                    5000     100.0  100.0  0.0275  0.0153  0.0192  0.0083   
       Beta         1000     100.0  100.0  0.0176  0.0180  0.0109  0.0113   
                    5000     100.0  100.0  0.0111  0.0124  0.0062  0.0071   

                         moment gap         quantile gap  
tool                         pyGLAM   GLDEX       median  
Family Distribution N                                     
FKML   Normal       1000     0.0000  0.0000       0.0000  
                    5000     0.0000  0.0000       0.0000  
       Gumbel       1000     0.0000  0.0980       0.0609  
                    5000     0.0000  0.1058       0.0640  
       Uniform      1000     0.0000  0.0119       0.0036  
                    5000     0.0000  0.0068       0.0021  
       Student-t    1000     0.0000  0.0000       0.0000  
                    5000     0.0000  0.0000       0.0000  
       Gamma        1000     0.0000  0.0848       0.0516  
                    5000     0.0000  0.0766       0.0461  
       Beta         1000     0.0000  0.0821       0.0413  
                    5000     0.0000  0.1073       0.0422  
RS     Normal       1000     0.0000  0.0000       0.0000  
                    5000     0.0000  0.0000       0.0000  
       Gumbel       1000     0.0000  0.0559       0.0443  
                    5000     0.0000  0.0000       0.0000  
       Uniform      1000     0.0001  0.0101       0.0038  
                    5000     0.0000  0.0072       0.0024  
       Student-t    1000     0.0000  0.0000       0.0000  
                    5000     0.0000  0.0000       0.0000  
       Gamma        1000     0.0000  0.0000       0.1197  
                    5000     0.0000  0.0000       0.1238  
       Beta         1000     0.0000  0.0000       0.0000  
                    5000     0.0000  0.0000       0.0000

mean  median    std
Family tool                        
FKML   GLDEX   1.317   1.090  0.718
       pyGLAM  0.393   0.169  0.686
RS     GLDEX   4.716   4.920  2.485
       pyGLAM  0.166   0.180  0.068

### 4.1 The RS fits of the Gamma with the full grid of starts

On the Gamma the RS moment equations have more than one exact root, and `GLDEX`, which searches
10,000 Sobol starting points, reaches one that `pyglam` with 15 starts does not. The fits of the
Gamma at $N = 5{,}000$ are repeated with the whole deterministic grid of 29 starts.

In [8]:
gamma = [s for s in samples if s['Distribution'] == 'Gamma' and s['N'] == max(sizes)]


def fit_rs(s, n):
    t0  = time.perf_counter()
    sol = glam.GlamRS().fit_lambdas(s['x'], method='least_squares', n_starts=n, seed=s['id'])
    el  = time.perf_counter() - t0
    cdf = targets['Gamma'].cdf(quantiles('RS', sol.x))
    return max(np.max(np.abs(cdf - U)), cdf[0], 1.0 - cdf[-1]), el


runs = {n: Parallel(n_jobs=n_jobs)(delayed(fit_rs)(s, n) for s in gamma) for n in (15, 29)}
gldex_gamma = scored[(scored['tool'] == 'GLDEX') & (scored['Family'] == 'RS') &
                     (scored['Distribution'] == 'Gamma') & (scored['N'] == max(sizes))]
full_grid = pd.DataFrame({f'pyGLAM, {n} starts': [np.median([k for k, _ in v]), np.median([t for _, t in v])]
                          for n, v in runs.items()}, index=['median KS', 'median time (s)'])
full_grid['GLDEX'] = [gldex_gamma['KS'].median(), gldex_gamma['time'].median()]
display(full_grid.round(4))

,"pyGLAM, 15 starts","pyGLAM, 29 starts",GLDEX
median KS,0.0275,0.0253,0.0153
median time (s),0.4041,0.8005,5.9450


## **5. LaTeX tables and raw results**

In [9]:
fig_dir = REPO / 'texto'
names = {'Normal': 'Normal', 'Gumbel': 'Gumbel', 'Uniform': 'Uniform',
         'Student-t': 'Student-$t$ ($\\nu=4$)', 'Gamma': 'Gamma', 'Beta': 'Beta'}


def fmt(v, d):
    return '---' if not np.isfinite(v) else f'{v:.{d}f}'


def sci(v):
    if not np.isfinite(v):
        return '---'
    if v == 0:
        return '$0$'
    mant, exp = f'{v:.1e}'.split('e')
    return f'${mant} \\times 10^{{{int(exp)}}}$'


# Part A: largest discrepancy over all parameter sets of each parameterization
lines = []
for family, d in functions.groupby('Family', sort=False):
    cols = ['quantile', 'cdf', 'pdf', 'mean', 'variance', 'skewness', 'kurtosis']
    lines.append(f'{family} & {len(d)} & ' + ' & '.join(sci(d[c].max()) for c in cols) + r' \\')
(fig_dir / 'z_tab_pyglam_r_functions_en.tex').write_text('\n'.join(lines).rstrip('\\ ') + '\n',
                                                         encoding='utf-8')

# Part B: one row per parameterization, target and sample size
lines = []
for family in classes:
    first = True
    for name in targets:
        for n in sizes:
            r = comparison.loc[(family, name, n)]
            cells = [fmt(r[('valid (%)', 'pyGLAM')], 0), fmt(r[('valid (%)', 'GLDEX')], 0),
                     fmt(r[('KS', 'pyGLAM')], 4), fmt(r[('KS', 'GLDEX')], 4),
                     fmt(r[('W1', 'pyGLAM')], 4), fmt(r[('W1', 'GLDEX')], 4),
                     fmt(r[('quantile gap', 'median')], 4)]
            label = names[name] if n == sizes[0] else ''
            lines.append(f"{family if first else ''} & {label} & {n:,} & " + ' & '.join(cells) + r' \\')
            first = False
    lines.append(r'\midrule')
body = '\n'.join(lines[:-1]).rstrip()
(fig_dir / 'z_tab_pyglam_r_fits_en.tex').write_text(body[:-2].rstrip() + '\n', encoding='utf-8')

functions.assign(**{'lambda': functions['lambda'].astype(str)}).to_csv(REPO / '01_pyglam_vs_gld_functions.csv', index=False)
scored.to_csv(REPO / '01_pyglam_vs_gld_fits.csv', index=False)
full_grid.to_csv(REPO / '01_pyglam_vs_gld_full_grid.csv')
print(timing)

                mean  median    std
Family tool                        
FKML   GLDEX   1.317   1.090  0.718
       pyGLAM  0.393   0.169  0.686
RS     GLDEX   4.716   4.920  2.485
       pyGLAM  0.166   0.180  0.068
